In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import setup_plotting, switch_cwd_to_root

switch_cwd_to_root()

import os

figure_dir = "figures/revision/supplement"
setup_plotting(figure_dir, display_dpi=300, save_dpi=300)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from tqdm.auto import tqdm

from spatial_tcr.clonal_expansion import (
    calc_empirical_p_values,
    filter_clonal_clusters_by_cell_type,
    identify_clonal_clusters,
    merge_clonal_clusters,
    remove_minority_avbv_expression,
)
from spatial_tcr.permutation import permute_positions
from spatial_tcr.tcr import get_tcr_genes

In [ ]:
data_dir = "data/xenium/processed"
path = f"{data_dir}/07.1-kidney_tcr_clones.h5ad"
adata = sc.read_h5ad(path)
adata

In [ ]:
av_genes, bv_genes = get_tcr_genes(adata)[0:2]

In [ ]:
# optionally remove minorty avbv expression per cell
remove_minority_avbv_expression(
    adata, av_genes, bv_genes, out_layer="avbv_clean", in_layer="counts"
)

In [ ]:
adata.X = adata.layers["avbv_clean"].copy()
# adata.X = adata.layers["counts"].copy()
# Get unique elements per row by iterating through each row
max_nunique_per_row_av = max(
    [len(np.unique(row)) for row in adata[:, av_genes].X.toarray()]
)
max_nunique_per_row_bv = max(
    [len(np.unique(row)) for row in adata[:, bv_genes].X.toarray()]
)
print(f"Max number of unique AV genes per cell: {max_nunique_per_row_av}")
print(f"Max number of unique BV genes per cell: {max_nunique_per_row_bv}")

In [ ]:
np.unique(adata[:, av_genes].X.toarray(), axis=1).shape

In [ ]:
ad_t = adata[adata.obs["cell_type_l1"] == "T"].copy()
ad_t.obs["cell_type_l2"].value_counts().sum()

In [ ]:
ad_t.obs["cell_type_l1.1"].value_counts().sum()

In [ ]:
ct_key = "cell_type_l2"
tcell_keys = [
    "CD4+",
    "Tregs",
    "MAIT",
    "CD8+",
    "NKT-like",
]


ct_key = "cell_type_l1.1"
tcell_keys = [
    "CD4+",
    "MAIT",
    "CD8+",
    "NKT-like",
]


clone_df = identify_clonal_clusters(
    adata,
    ct_key=ct_key,
    av_genes=av_genes,
    bv_genes=bv_genes,
    sample_key="cc",
    tcell_keys=tcell_keys,
    max_dist=100,
    min_cells=2,
    layer="avbv_clean",
)

In [ ]:
avbv_cluster = merge_clonal_clusters(clone_df)
adata.obs["avbv_cluster"] = None
adata.obs.loc[avbv_cluster.index, "avbv_cluster"] = avbv_cluster

In [ ]:
filter_clonal_clusters_by_cell_type(
    adata,
    ct_key="cell_type_l1.1",
    prohibited_combinations=[("CD4+", "CD8+")],
    in_key="avbv_cluster",
    out_key="avbv_cluster_filtered",
    verbose=1,
)

In [ ]:
cc_key = "avbv_cluster_filtered"
# cc_key = "avbv_cluster"

for sample in adata.obs["sample"].unique()[1:2]:
    ad_sub = adata[adata.obs["sample"] == sample].copy()
    if ad_sub.obs[cc_key].notna().sum() == 0:
        continue
    sc.pl.spatial(ad_sub, color=cc_key, show=True, spot_size=20)

In [ ]:
adata.X = adata.layers["counts"].copy()

In [ ]:
adata.write_h5ad(f"{data_dir}/08.1-kidney_tcr_clonal_clusters.h5ad")

## Random permutation test

permute full expression per cell

In [ ]:
seed = 42
rng = np.random.RandomState(seed)
n_perms = 1000
layer = "avbv_clean"
# layer = "counts"
save_path = f"results/revision/clonal-expansion/clonal_clusters_simulated_postionwise_{layer}.csv"

if not os.path.exists(save_path):
    ad_tmp = adata[adata.obs[ct_key].isin(tcell_keys), av_genes + bv_genes].copy()

    cluster_results = []

    for _ in tqdm(range(n_perms)):
        ad_perm = ad_tmp.copy()

        ad_perm = permute_positions(ad_perm, rng=rng, sample_key="cc")

        # randomly scatter gene expression around t cells
        # ad_perm = permute_gene_expression(ad_perm, rng=rng, sample_key="cc")

        clone_df = identify_clonal_clusters(
            ad_perm,
            av_genes=av_genes,
            bv_genes=bv_genes,
            sample_key="cc",
            max_dist=100,
            min_cells=2,
            verbose=False,
            layer=layer,
        )
        avbv_cluster = merge_clonal_clusters(clone_df, verbose=False)
        ad_perm.obs["avbv_cluster"] = None
        ad_perm.obs.loc[avbv_cluster.index, "avbv_cluster"] = avbv_cluster
        filter_clonal_clusters_by_cell_type(
            ad_perm,
            ct_key="cell_type_l1.1",
            prohibited_combinations=[("CD4+", "CD8+")],
            in_key="avbv_cluster",
            out_key="avbv_cluster_filtered",
            prog_bar=False,
        )
        cluster_results.append(ad_perm.obs["avbv_cluster_filtered"])

    cluster_results = pd.concat(cluster_results, axis=1)
    cluster_results.to_csv(save_path)

In [ ]:
layer = "avbv_clean"
save_path = f"results/revision/clonal-expansion/clonal_clusters_simulated_postionwise_{layer}.csv"
cluster_results = pd.read_csv(save_path, index_col=0)
# cluster_results.nunique().values

In [ ]:
n_perms = 1000
counts_sim = cluster_results.nunique().values
counts_obs = adata.obs["avbv_cluster_filtered"].nunique()
z_scores, p_values, q_values = calc_empirical_p_values(counts_obs, counts_sim, n_perms)
print(p_values)
print(q_values)

In [ ]:
sns.set_theme(style="ticks", context="paper")
fig, ax = plt.subplots(figsize=(6, 4))
hist = sns.histplot(counts_sim, ax=ax)
ax.set_xlabel("number of clonal clusters", fontsize=10)
ax.set_ylabel("count", fontsize=10)
ax.set_title("clonal clusters in simulated data", fontsize=10)
ax.tick_params(axis="both", labelsize=8)

# Get the bar coordinates
# for p in hist.patches:
#     print(f"Bar center: {p.get_x() + p.get_width() / 2:.2f}, height: {p.get_height()}")

bar_centers = [p.get_x() + p.get_width() / 2 for p in hist.patches]
print(bar_centers)

# draw red vertical line for observed number of clonal clusters
ax.axvline(counts_obs, color="red", linestyle="--")
# add text for observed number of clonal clusters
ax.text(
    counts_obs - 1,
    125,
    f"observed number of clonal clusters: {counts_obs}\npermutation p-value: {np.round(p_values, 5)}",
    color="red",
    ha="right",
    fontsize=8,
)

# split x axis into two parts
sns.despine(ax=ax)

plt.savefig(
    os.path.join(figure_dir, "clonal_clusters_simulated_postionwise_avbv_clean.pdf"),
    dpi=300,
    bbox_inches="tight",
    transparent=True,
)
# plt.close()